### Ingestión la carpeta movie language

In [0]:
%run '../includes/configuration'

In [0]:
%run '../includes/common_functions'

In [0]:
dbutils.widgets.text("p_environment","")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date","2024-12-23")
v_file_date = dbutils.widgets.get("p_file_date")

##### Paso 1. Leer los archivos

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
movies_languages_schema = StructType(fields=[
    StructField("movieId", IntegerType(), True),
    StructField("languageId", IntegerType(), True),
    StructField("languageRoleId", IntegerType(), True)
])

In [0]:
movies_languages_df = spark.read \
                .schema(movies_languages_schema) \
                .option("multiline",True) \
                .json(f"{bronze_folder_path}/{v_file_date}/movie_language/*.json")

##### Paso 2 - Renombrar las columnas y añadir nuevas columnas

In [0]:
from pyspark.sql.functions import col, concat, current_timestamp, lit, trim

In [0]:
movies_languages_with_columns_df = movies_languages_df \
                        .withColumnRenamed("movieId","movie_id") \
                        .withColumnRenamed("languageId","language_id") \
                        .withColumn("ingestion_date", current_timestamp()) \
                        .withColumn("environment", lit(v_environment)) \
                        .withColumn("file_date", lit(v_file_date))

#####Paso 3 - Eliminar la columna languageRoleId

In [0]:
from pyspark.sql.functions import col

In [0]:
movies_languages_final_df = movies_languages_with_columns_df.drop(col("languageRoleId"))

#####Paso 4 - Escribir la salida en formato parquet

In [0]:
#overwrite_partition("movie_silver","movies_languages","file_date",v_file_date)

In [0]:
merge_condition = "tgt.movie_id = src.movie_id AND tgt.language_id = src.language_id AND tgt.file_date = src.file_date"
merge_delta_lake(movies_languages_final_df,"movie_silver","movies_languages",merge_condition,"file_date")

In [0]:
%sql 
SELECT file_date, COUNT(1) FROM movie_silver.movies_languages
GROUP BY file_date 

file_date,count(1)
2024-12-16,6000
2024-12-23,3010
2024-12-30,2730
